# Lab 2.1 &mdash; Chain-of-Thought, Built as a Chain

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Compose a real LCEL chain: <code>prompt | model | parser</code>
- Build two arms that differ in <i>one</i> string and nothing else
- Write the verdict down first, then run the eval set with <code>.batch()</code>

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 2 labs work one case: an internal employee help desk.
> The rules are ordinary on purpose &mdash; the only new thing here is how the agent reasons.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# An internal employee help desk. Ordinary rules on purpose: the only new thing in these five
# labs is LangChain. Nothing here is real data and nothing leaves this notebook.

REQUESTS = {
    "EHD-7001": {"who": "Priya Nair",   "category": "access",   "urgency": "high",
                 "wants": "reset",
                 "text": "Locked out of the payroll portal after the password reset."},
    "EHD-7002": {"who": "Rahul Menon",  "category": "hardware", "urgency": "high",
                 "wants": "replacement",
                 "text": "Laptop battery has swollen and the case is bulging."},
    "EHD-7003": {"who": "Anita Sharma", "category": "software", "urgency": "low",
                 "wants": "licence",
                 "text": "Need a licence for the diagramming tool, about 180 USD a year."},
    "EHD-7004": {"who": "Vikram Rao",   "category": "access",   "urgency": "medium",
                 "wants": "admin-rights",
                 "text": "Please give me admin rights on the finance reporting system."},
    "EHD-7005": {"who": "Priya Nair",   "category": "hardware", "urgency": "low",
                 "wants": "replacement",
                 "text": "Second monitor flickers every few minutes."},
}

# The handbook, one entry per category. Every judgement in this module comes from these.
HANDBOOK = {
    "access":   "Verify identity, then reset. The help desk NEVER grants elevated or admin "
                "rights -- route those to Identity and Access Management.",
    "hardware": "Replace under warranty. A swollen battery is a safety issue: stop use "
                "immediately and replace the same day, whatever urgency the employee set.",
    "software": "Licences over 100 USD per year need the cost-centre owner's approval first.",
}

SLA_HOURS = {"high": 4, "medium": 24, "low": 72}
ROUTE_OUT = {"admin-rights"}     # what the help desk must hand to another team, never do itself

print(f"{len(REQUESTS)} help desk requests, {len(HANDBOOK)} handbook entries loaded")

## Concept

**Chain-of-thought** asks the model to show its working before it answers. It usually helps on
tasks that combine two facts, and it always costs tokens. How much of each is a property of
*your* task, not a fact about language models &mdash; so measure it on this one.

The thing you measure it with is worth as much as the answer. **LCEL** composes a prompt, a
model and a parser into one runnable with `|`:

```python
chain = prompt | model | StrOutputParser()
chain.invoke({...})     # one case
chain.batch([{...}, {...}, ...])   # the whole eval set, concurrently
```

Everything later in this course is built this way.

## Section 1 &mdash; The chain, and the one string that makes it chain-of-thought

Three stages, one pipe. `ChatPromptTemplate` turns variables into messages, the model answers,
`StrOutputParser` pulls `.content` out so the chain returns a plain string.

The two arms share the human message, the model, the parser and the case file. The *only*
difference is the system instruction &mdash; which is what makes this a measurement rather than
an anecdote. Both candidate instructions are written out below.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

HUMAN = ("REQUEST {rid}: {text}\n"
         "CATEGORY: {category}   URGENCY SET BY THE EMPLOYEE: {urgency}\n"
         "HANDBOOK: {rule}\n\n"
         "What must the help desk do next?")


def instruction(mode: str) -> str:
    """The only difference between the two arms."""
    answer_only  = ("You are an employee help desk analyst. Reply with the single next "
                    "action and nothing else.")
    show_working = ("You are an employee help desk analyst. First restate the handbook rule, "
                    "then the urgency the employee set, then say which of the two decides the "
                    "timing. Finish with one line beginning 'ACTION:'.")

    if mode == "direct":
        return BLANK        # TODO: which one asks for the answer with no working shown?
    return BLANK            # TODO: and which one is chain-of-thought?


def build_prompt(mode: str) -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages([("system", instruction(mode)), ("human", HUMAN)])


def build_chain(mode: str, model):
    """prompt | model | parser. `model` is any Runnable -- a chat model is only one kind."""
    return build_prompt(mode) | model | StrOutputParser()


def case_vars(rid: str) -> dict:
    """One request, flattened into the five template variables."""
    r = REQUESTS[rid]
    return {"rid": rid, "text": r["text"], "category": r["category"],
            "urgency": r["urgency"], "rule": HANDBOOK[r["category"]]}

In [ ]:
# --- Self-check: Section 1   (a real chain, really invoked -- with a stub where the model goes)
STUB = RunnableLambda(lambda messages: AIMessage(content="ACTION: replace it the same day"))

def sys_text(mode):
    return build_prompt(mode).format_messages(**case_vars("EHD-7002"))[0].content

def human_text(mode):
    return build_prompt(mode).format_messages(**case_vars("EHD-7002"))[1].content

check("the template renders a system message and a human message",
      lambda: [m.type for m in build_prompt("cot").format_messages(**case_vars("EHD-7002"))]
              == ["system", "human"])
check("the request text is substituted in, not left as a brace",
      lambda: "swollen" in human_text("cot") and "{text}" not in human_text("cot"))
check("the two arms differ in the system message ONLY",
      lambda: human_text("direct") == human_text("cot") and sys_text("direct") != sys_text("cot"),
      "if anything else differs, the comparison measures that instead")
check("the direct arm asks for the action and no working",
      lambda: "nothing else" in sys_text("direct"))
check("the chain-of-thought arm asks for the working first",
      lambda: "restate" in sys_text("cot") and "ACTION:" in sys_text("cot"))
check("prompt | model | parser returns a plain string, not an AIMessage",
      lambda: isinstance(build_chain("cot", STUB).invoke(case_vars("EHD-7002")), str),
      "StrOutputParser is the third stage -- drop it and you get a message object back")
score()

## Section 2 &mdash; The eval set, and the verdict you write down first

Five requests, and what a correct answer has to mention. Keyword lists rather than exact
wording, because you are grading the decision, not the prose.

Four of the five are settled. **EHD-7002 is the one that carries the lesson.** Rahul set the
urgency to `high`, which is a 4-hour SLA. The handbook says a swollen battery is a safety issue:
replace it *the same day, whatever urgency the employee set*. Four hours is inside the same day,
so an answer that says &ldquo;within 4 hours per the high SLA&rdquo; lands on a timing that
happens to be acceptable &mdash; by reading the row the handbook told it to ignore.

Decide what you will accept **before** you see a single model output. That is the difference
between an evaluation and a story about one run.

In [ ]:
def expected_for(rid: str) -> list:
    """What a correct answer must mention, lower-cased. Substrings, so wording is free."""
    settled = {
        "EHD-7001": ["reset"],
        "EHD-7003": ["approval"],
        "EHD-7004": ["identity and access"],
        "EHD-7005": ["replace"],
    }
    sla_wins  = ["within 4 hours"]      # the urgency the employee set
    rule_wins = ["same day"]            # the handbook's safety rule

    if rid == "EHD-7002":
        return BLANK        # TODO: which one is the correct answer for a swollen battery?
    return settled[rid]


def graded(rid: str, answer: str) -> bool:
    """Correct if the answer mentions everything that case requires."""
    text = (answer or "").lower()
    return all(k in text for k in expected_for(rid))


def pass_count(answers: dict) -> int:
    return sum(1 for rid, a in answers.items() if graded(rid, a))

In [ ]:
# --- Self-check: Section 2   (hand-written answers, graded offline -- no model)
HAND = {
    "EHD-7001": "Verify identity, then reset the payroll portal password.",
    "EHD-7002": "Stop use and replace the laptop the same day.",
    "EHD-7003": "The cost-centre owner's approval is needed first -- 180 USD is over the limit.",
    "EHD-7004": "Route to Identity and Access Management; the help desk never grants admin rights.",
    "EHD-7005": "Replace the monitor under warranty.",
}

check("a same-day answer for the swollen battery is correct",
      lambda: graded("EHD-7002", "Stop use and replace the laptop the same day."))
check("'within 4 hours per the high SLA' is NOT accepted",
      lambda: not graded("EHD-7002", "Replace within 4 hours per the high SLA."),
      "the timing is fine and the reasoning is wrong -- it read the row the handbook overrides")
check("granting admin rights is not an acceptable answer to EHD-7004",
      lambda: not graded("EHD-7004", "Grant admin rights on the finance reporting system."))
check("five hand-written correct answers score 5/5",
      lambda: pass_count(HAND) == 5,
      "if this is not 5/5 the bar is grading wording rather than decisions")
score()

## Run it for real

Two arms, five cases, one `.batch()` each.

In [ ]:
CASES = [case_vars(rid) for rid in sorted(REQUESTS)]

def run_arm(mode: str) -> dict:
    outs = build_chain(mode, get_llm()).batch(CASES)
    return {c["rid"]: o for c, o in zip(CASES, outs)}

def bake_off():
    for mode in ["direct", "cot"]:
        answers = run_arm(mode)
        print(f"=== {mode}: {pass_count(answers)}/{len(CASES)} ===")
        for rid, a in answers.items():
            last = (a.strip().splitlines() or [""])[-1]
            print(f'  {"ok " if graded(rid, a) else "NO "}{rid}: {last[:88]}')
        print()

if llm_ready():
    guard(bake_off)

### Read it

Ten model calls went out in two `.batch()` calls, and `.batch()` ran each set **concurrently**
&mdash; that is the whole reason an eval set is a thing you run every time you touch a prompt
rather than a thing you promise to do later.

Look at EHD-7002 in both arms. The direct arm tends to answer from the loudest field on the page,
which is `URGENCY: high`; the chain-of-thought arm is forced to put the handbook rule down first,
and once the rule is written out the override is hard to miss. Nothing changed but one string.

The bar decided that. &ldquo;Within 4 hours&rdquo; is a defensible answer to a human reader and
you would have accepted it, silently, if you had written the bar after seeing it.

In [ ]:
score()

## Your turn

1. Add a third arm whose instruction just says &ldquo;think step by step&rdquo; and nothing about
   the handbook. Does the generic phrase buy what the ordered procedure buys?
2. Change `expected_for("EHD-7002")` to `sla_wins` and re-run. Both arms will look better. Say
   out loud what you gave up to get that number.